# 3단계 v4. 머신러닝 분석·학습·평가

`구축 데이터셋_v3`를 이용해 정상 금융상담과 보이스피싱을 비교하고, 보이스피싱 유형 분류·구간 탐지·유사 사건 군집화를 수행합니다.

v4 추가 사항
- 정상상담과 보이스피싱의 텍스트 길이 차이 점검
- 학습 데이터 1:3과 1:1 비율 비교
- 긴 정상상담을 짧은 구간으로 맞춘 길이 보정 실험
- 금액의 방향(피해자에게 제시/피해자에게 요구/단순 언급)과 용도 EDA
- 모델 선택은 검증 데이터로 하고 최종 테스트는 한 번만 실행


In [ ]:
# 0. 라이브러리 설치
!pip -q install pandas pyarrow scikit-learn seaborn matplotlib koreanize-matplotlib joblib openpyxl

In [ ]:
# 1. 라이브러리 불러오기 및 Google Drive 연결
from google.colab import drive
from pathlib import Path
from IPython.display import display
import hashlib, json, re, unicodedata, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
from sklearn.base import clone
from sklearn.cluster import AgglomerativeClustering, KMeans, MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (accuracy_score, adjusted_rand_score, average_precision_score,
    classification_report, confusion_matrix, precision_recall_fscore_support, silhouette_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
warnings.filterwarnings('ignore')
drive.mount('/content/drive')
print('Google Drive 연결 완료')

## 1. 경로와 설정값

In [ ]:
# 2. 경로와 공통 설정
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / '보이스피싱_분석'
DATASET_ROOT = PROJECT_ROOT / '구축 데이터셋_v3'
STANDARD_ROOT = DATASET_ROOT / '01_standard_tables'
ML_ROOT = DATASET_ROOT / '02_ml_tables'
OUTPUT_ROOT = PROJECT_ROOT / '머신러닝_분석결과_v4'
KOREAN_ROOT = OUTPUT_ROOT / '00_한글_확인용'
EDA_ROOT = OUTPUT_ROOT / '01_EDA'
SPLIT_ROOT = OUTPUT_ROOT / '02_데이터분리'
MODEL_ROOT = OUTPUT_ROOT / '03_모델'
PRED_ROOT = OUTPUT_ROOT / '04_예측결과'
REPORT_ROOT = OUTPUT_ROOT / '05_보고서'
for folder in [KOREAN_ROOT, EDA_ROOT, SPLIT_ROOT, MODEL_ROOT, PRED_ROOT, REPORT_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)
SEED = 42
TEST_RATIO = 0.20
DEV_RATIO = 0.20
MAX_FEATURES = 50000
MIN_TEXT_LENGTH = 10
assert ML_ROOT.exists(), f'1단계 결과 경로를 확인하세요: {ML_ROOT}'
print('입력:', DATASET_ROOT)
print('출력:', OUTPUT_ROOT)

## 2. 분석 시나리오

In [ ]:
# 3. 분석 목적과 평가 지표
scenario_df = pd.DataFrame([
    ['정상상담 vs 보이스피싱', '이진 분류', 'Recall·F1·PR-AUC'],
    ['보이스피싱 유형', '대출사기형 vs 수사기관사칭형', 'Macro F1·혼동행렬'],
    ['전체·부분 구간 탐지', '대화 어느 구간에서도 탐지 가능한지 확인', '구간별 Recall·F1·PR-AUC'],
    ['유사 사건', '정답 없이 비슷한 사건 묶기', 'Silhouette·seed 안정성'],
    ['길이·비율 민감도', '출처와 길이 차이에 의한 편향 확인', '검증 Recall·F1·PR-AUC'],
], columns=['시나리오','목적','평가지표'])
display(scenario_df)
scenario_df.to_csv(REPORT_ROOT/'분석_시나리오.csv', index=False, encoding='utf-8-sig')
print('실제 피해 여부 정답이 없으므로 피해 발생 확률은 학습하지 않습니다.')

## 3. 데이터 불러오기와 전처리

In [ ]:
# 4. 1단계 데이터 불러오기
def read_table(folder, name):
    parquet_path = folder / f'{name}.parquet'
    csv_path = folder / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    assert csv_path.exists(), f'테이블을 찾지 못했습니다: {name}'
    return pd.read_csv(csv_path, encoding='utf-8-sig')
detection_df = read_table(ML_ROOT, 'fraud_detection_ml')
type_df = read_table(ML_ROOT, 'fraud_type_ml')
segment_df = read_table(ML_ROOT, 'segment_detection_ml')
cluster_df = read_table(ML_ROOT, 'case_clustering_ml')
amount_df = read_table(STANDARD_ROOT, 'vp_amount_events')
summary_df = pd.DataFrame([
    ['정상·사기', len(detection_df), len(detection_df.columns)],
    ['사기유형', len(type_df), len(type_df.columns)],
    ['전체·부분구간', len(segment_df), len(segment_df.columns)],
    ['사건군집', len(cluster_df), len(cluster_df.columns)],
    ['금액이벤트', len(amount_df), len(amount_df.columns)],
], columns=['데이터','행수','컬럼수'])
display(summary_df)
required_amount = {'amount_krw','amount_status','amount_direction','amount_purpose'}
assert required_amount.issubset(amount_df.columns), f'v3 금액 컬럼 누락: {required_amount-set(amount_df.columns)}'

In [ ]:
# 5. 텍스트 정리: 원문은 보존하고 clean_text를 별도로 만듭니다.
def clean_text(text):
    text = unicodedata.normalize('NFKC', str(text or '')).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'(?m)^\s*(tx|rx|화자\s*\d*|범인|피해자)\s*[:：]\s*', ' ', text)
    text = re.sub(r'[*#xX]{2,}', ' 마스킹 ', text)
    text = re.sub(r'\b\d{2,}\b', ' 숫자 ', text)
    return re.sub(r'\s+', ' ', text).strip()
def prepare(df, text_col):
    result = df.copy()
    result['clean_text'] = result[text_col].fillna('').map(clean_text)
    result = result[result['clean_text'].str.len() >= MIN_TEXT_LENGTH].copy()
    result['text_hash'] = result['clean_text'].map(lambda x: hashlib.sha256(x.encode()).hexdigest())
    return result.reset_index(drop=True)
def remove_conflicts_and_duplicates(df, label):
    conflict = df.groupby('text_hash')[label].nunique()
    conflict_hashes = set(conflict[conflict > 1].index)
    result = df[~df['text_hash'].isin(conflict_hashes)].copy()
    duplicate_count = int(result.duplicated([label,'text_hash']).sum())
    return result.drop_duplicates([label,'text_hash']).reset_index(drop=True), len(conflict_hashes), duplicate_count
detection_df = prepare(detection_df, 'model_input_text')
type_df = prepare(type_df, 'model_input_text')
segment_df = prepare(segment_df, 'window_text')
cluster_df = prepare(cluster_df, 'model_input_text')
detection_df, d_conflicts, d_duplicates = remove_conflicts_and_duplicates(detection_df, 'fraud_label')
type_df, t_conflicts, t_duplicates = remove_conflicts_and_duplicates(type_df, 'supervised_target')
amount_df['amount_krw'] = pd.to_numeric(amount_df['amount_krw'], errors='coerce')
amount_df['amount_10k_krw'] = amount_df['amount_krw'] / 10000
amount_df['log_amount_10k_krw'] = np.log1p(amount_df['amount_10k_krw'].clip(lower=0))
display(pd.DataFrame([['정상·사기',d_conflicts,d_duplicates,len(detection_df)],
                      ['사기유형',t_conflicts,t_duplicates,len(type_df)]],
                     columns=['데이터','라벨충돌해시','정확중복제외','최종행수']))

## 4. 사람이 확인하는 한글 컬럼 데이터

In [ ]:
# 6. 분석 내부는 영문 컬럼, 확인용 CSV는 한글 컬럼으로 저장합니다.
column_ko = {
 'conversation_id':'대화ID','case_id':'사건ID','file_id':'원본파일ID','group_id':'분리그룹ID',
 'fraud_label':'사기여부','supervised_target':'보이스피싱유형','source_group':'데이터출처',
 'sample_scope':'표본범위','window_position':'구간위치','model_input_text':'모델입력원문',
 'window_text':'구간원문','clean_text':'정제텍스트','original_split':'원본데이터구분',
 'amount_krw':'금액_원','amount_10k_krw':'금액_만원','amount_text':'금액원문',
 'amount_status':'금액상태','amount_direction':'금액방향','amount_purpose':'금액용도',
 'amount_direction_evidence':'금액방향_근거문장','amount_direction_confidence':'금액방향_신뢰도',
 'evidence_text':'근거발화','evidence_role':'근거화자역할'
}
def save_korean(df, filename):
    result = df.rename(columns=column_ko)
    result.to_csv(KOREAN_ROOT/filename, index=False, encoding='utf-8-sig')
    return result
save_korean(detection_df, '정상상담_보이스피싱_분류데이터.csv')
save_korean(type_df, '보이스피싱_유형분류데이터.csv')
save_korean(segment_df, '전체_부분구간_탐지데이터.csv')
save_korean(cluster_df, '유사사건_군집데이터.csv')
amount_ko = save_korean(amount_df, '금액이벤트_방향_용도_만원.csv')
display(amount_ko.head(5))
print('한글 확인용 CSV 저장:', KOREAN_ROOT)

## 5. EDA와 편향 점검

In [ ]:
# 7. 클래스·텍스트 길이·금액 방향 확인
detection_df['text_length'] = detection_df['clean_text'].str.len()
type_df['text_length'] = type_df['clean_text'].str.len()
length_summary = detection_df.groupby('fraud_label')['text_length'].agg(['count','mean','median','std','min','max']).reset_index()
display(length_summary)
length_summary.to_csv(EDA_ROOT/'텍스트길이_요약.csv', index=False, encoding='utf-8-sig')
fig, axes = plt.subplots(1,3,figsize=(18,5))
sns.countplot(data=detection_df,x='fraud_label',ax=axes[0]); axes[0].set_title('정상상담과 보이스피싱 분포')
sns.boxplot(data=detection_df,x='fraud_label',y='text_length',showfliers=False,ax=axes[1]); axes[1].set_title('정제 텍스트 길이')
sns.countplot(data=type_df,x='supervised_target',ax=axes[2]); axes[2].set_title('보이스피싱 유형 분포')
for ax in axes: ax.tick_params(axis='x',rotation=15)
plt.tight_layout(); plt.savefig(EDA_ROOT/'기본_EDA.png',dpi=160,bbox_inches='tight'); plt.show()
direction_summary = amount_df['amount_direction'].fillna('UNKNOWN').value_counts().rename_axis('금액방향').reset_index(name='건수')
purpose_summary = amount_df['amount_purpose'].fillna('UNKNOWN').value_counts().rename_axis('금액용도').reset_index(name='건수')
display(direction_summary); display(purpose_summary.head(15))
direction_summary.to_csv(EDA_ROOT/'금액방향_분포.csv',index=False,encoding='utf-8-sig')
purpose_summary.to_csv(EDA_ROOT/'금액용도_분포.csv',index=False,encoding='utf-8-sig')
fig,axes=plt.subplots(1,2,figsize=(16,5))
sns.countplot(data=amount_df,x='amount_direction',order=amount_df['amount_direction'].value_counts().index,ax=axes[0])
axes[0].set_title('금액 방향 분포'); axes[0].tick_params(axis='x',rotation=25)
top_purpose=amount_df['amount_purpose'].value_counts().head(10).index
sns.countplot(data=amount_df[amount_df['amount_purpose'].isin(top_purpose)],y='amount_purpose',order=top_purpose,ax=axes[1])
axes[1].set_title('금액 용도 상위 10개')
plt.tight_layout(); plt.savefig(EDA_ROOT/'금액방향_용도.png',dpi=160,bbox_inches='tight'); plt.show()
upper=amount_df['amount_10k_krw'].quantile(.99)
plot_amount=amount_df[amount_df['amount_10k_krw'].between(0,upper)]
plt.figure(figsize=(11,5)); sns.histplot(data=plot_amount,x='amount_10k_krw',hue='amount_direction',bins=30)
plt.xlabel('금액(만원)'); plt.title('금액 방향별 분포: 상위 1% 이상치 제외')
plt.tight_layout(); plt.savefig(EDA_ROOT/'금액방향별_금액분포_만원.png',dpi=160); plt.show()

## 6. 학습·검증·최종 테스트 분리

같은 원본 통화는 반드시 하나의 세트에만 들어갑니다. 정상상담의 공식 Validation은 최종 테스트로 보존하고, 보이스피싱은 `group_id` 단위로 대응 분리합니다.

In [ ]:
# 8. 그룹 누출 없는 분리 함수
def random_group_parts(groups, test_ratio=TEST_RATIO, dev_ratio=DEV_RATIO):
    groups=np.array(sorted(pd.Series(groups).dropna().astype(str).unique()))
    train_dev,test=train_test_split(groups,test_size=test_ratio,random_state=SEED)
    train,dev=train_test_split(train_dev,test_size=dev_ratio,random_state=SEED)
    result={g:'TRAIN' for g in train}; result.update({g:'DEV' for g in dev}); result.update({g:'TEST' for g in test})
    return result
def stratified_group_parts(df,label_col):
    grouped=df.groupby('group_id')[label_col].agg(lambda s:s.mode().iloc[0]).reset_index()
    assert df.groupby('group_id')[label_col].nunique().max()==1
    train_dev,test=train_test_split(grouped,test_size=TEST_RATIO,random_state=SEED,stratify=grouped[label_col])
    train,dev=train_test_split(train_dev,test_size=DEV_RATIO,random_state=SEED,stratify=train_dev[label_col])
    result={g:'TRAIN' for g in train.group_id}; result.update({g:'DEV' for g in dev.group_id}); result.update({g:'TEST' for g in test.group_id})
    return result
def check_split(df,label_col):
    assert df.groupby('group_id')['ml_split'].nunique().max()==1, '같은 원본 통화가 여러 세트에 섞였습니다.'
    assert {'TRAIN','DEV','TEST'}.issubset(set(df.ml_split))
    for name in ['TRAIN','DEV','TEST']:
        assert df.loc[df.ml_split.eq(name),label_col].nunique()>=2, f'{name}에 클래스가 하나뿐입니다.'
    return df.groupby(['ml_split',label_col]).size().reset_index(name='건수')

In [ ]:
# 9. 정상 vs 사기 분리
det=detection_df.copy(); det['group_id']=det['group_id'].astype(str); det['ml_split']=''
normal_mask=det.fraud_label.eq('LEGITIMATE_FINANCIAL_CALL')
split_text=det.get('original_split',pd.Series('',index=det.index)).fillna('').astype(str).str.upper()
normal_test=normal_mask & split_text.str.contains('VALID|TEST')
normal_train_pool=normal_mask & ~normal_test
# 공식 Validation 표기가 없을 때만 정상상담도 원본 그룹 단위로 대응 분리합니다.
if normal_test.sum()==0:
    fallback_map=random_group_parts(det.loc[normal_mask,'group_id'].unique())
    det.loc[normal_mask,'ml_split']=det.loc[normal_mask,'group_id'].map(fallback_map)
    normal_train_pool=pd.Series(False,index=det.index)
if normal_train_pool.any():
    normal_train_groups=det.loc[normal_train_pool,'group_id'].unique()
    normal_train,normal_dev=train_test_split(normal_train_groups,test_size=DEV_RATIO,random_state=SEED)
    det.loc[normal_train_pool & det.group_id.isin(normal_train),'ml_split']='TRAIN'
    det.loc[normal_train_pool & det.group_id.isin(normal_dev),'ml_split']='DEV'
    det.loc[normal_test,'ml_split']='TEST'
fraud_groups=det.loc[~normal_mask,'group_id'].unique()
fraud_map=random_group_parts(fraud_groups)
det.loc[~normal_mask,'ml_split']=det.loc[~normal_mask,'group_id'].map(fraud_map)
assert not det.ml_split.eq('').any(), '분리되지 않은 데이터가 있습니다.'
split_summary=check_split(det,'fraud_label'); display(split_summary)
split_summary.to_csv(SPLIT_ROOT/'정상사기_분리요약.csv',index=False,encoding='utf-8-sig')
det[['conversation_id','group_id','fraud_label','ml_split']].to_csv(SPLIT_ROOT/'정상사기_split_id.csv',index=False,encoding='utf-8-sig')

## 7. 길이·비율 편향 민감도 실험

In [ ]:
# 10. 학습용 1:3·1:1 표본과 길이 보정 표본 만들기
FRAUD='VOICE_PHISHING'; NORMAL='LEGITIMATE_FINANCIAL_CALL'
def sample_ratio(df, normal_per_fraud):
    fraud=df[df.fraud_label.eq(FRAUD)]
    normal=df[df.fraud_label.eq(NORMAL)]
    n=min(len(normal),len(fraud)*normal_per_fraud)
    normal=normal.sample(n=n,random_state=SEED) if n else normal
    return pd.concat([fraud,normal]).sample(frac=1,random_state=SEED).reset_index(drop=True)
def center_window(text,target_chars):
    text=str(text)
    if len(text)<=target_chars: return text
    start=(len(text)-target_chars)//2
    return text[start:start+target_chars]
train_all=det[det.ml_split.eq('TRAIN')].copy()
dev_original=det[det.ml_split.eq('DEV')].copy()
target_chars=int(train_all.loc[train_all.fraud_label.eq(FRAUD),'text_length'].median())
target_chars=int(np.clip(target_chars,250,1500))
train_1to3=sample_ratio(train_all,3)
train_1to1=sample_ratio(train_all,1)
train_length_1to1=train_1to1.copy(); dev_length=dev_original.copy()
train_length_1to1['clean_text']=train_length_1to1.clean_text.map(lambda x:center_window(x,target_chars))
dev_length['clean_text']=dev_length.clean_text.map(lambda x:center_window(x,target_chars))
experiment_summary=pd.DataFrame([
 ['원본_1대3',len(train_1to3),train_1to3.fraud_label.value_counts().to_dict(),False],
 ['원본_1대1',len(train_1to1),train_1to1.fraud_label.value_counts().to_dict(),False],
 ['길이보정_1대1',len(train_length_1to1),train_length_1to1.fraud_label.value_counts().to_dict(),True],
],columns=['실험','학습행수','클래스분포','길이보정'])
display(experiment_summary); print('길이 보정 기준:',target_chars,'자')
experiment_summary.to_csv(EDA_ROOT/'비율_길이보정_실험설계.csv',index=False,encoding='utf-8-sig')

## 8. 모델 비교 공통 함수

In [ ]:
# 11. 후보 모델과 평가 함수
def binary_candidates():
    return {
      'Dummy':Pipeline([('tfidf',TfidfVectorizer(max_features=1000)),('model',DummyClassifier(strategy='prior'))]),
      'Word_TFIDF_Logistic':Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),('model',LogisticRegression(max_iter=1500,class_weight='balanced',random_state=SEED))]),
      'Char_TFIDF_Logistic':Pipeline([('tfidf',TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),('model',LogisticRegression(max_iter=1500,class_weight='balanced',random_state=SEED))]),
      'Char_TFIDF_LinearSVM':Pipeline([('tfidf',TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),('model',LinearSVC(class_weight='balanced',random_state=SEED))]),
      'Char_TFIDF_SGD':Pipeline([('tfidf',TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),('model',SGDClassifier(loss='log_loss',class_weight='balanced',random_state=SEED))]),
      'Word_TFIDF_ComplementNB':Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),('model',ComplementNB())])}
def positive_score(model,x,positive):
    classes=list(model.classes_); idx=classes.index(positive)
    if hasattr(model,'predict_proba'): return model.predict_proba(x)[:,idx]
    score=model.decision_function(x)
    return score if np.ndim(score)==1 and idx==1 else (-score if np.ndim(score)==1 else score[:,idx])
def binary_scores(y_true,y_pred,score,positive=FRAUD):
    p,r,f1,_=precision_recall_fscore_support(y_true,y_pred,average='binary',pos_label=positive,zero_division=0)
    y_bin=(np.asarray(y_true)==positive).astype(int)
    return {'accuracy':accuracy_score(y_true,y_pred),'precision':p,'recall':r,'f1':f1,'pr_auc':average_precision_score(y_bin,score)}
def compare_binary(train,dev,label='fraud_label'):
    rows=[]
    for name,model in binary_candidates().items():
        model.fit(train.clean_text,train[label]); pred=model.predict(dev.clean_text); score=positive_score(model,dev.clean_text,FRAUD)
        rows.append({'모델':name,**binary_scores(dev[label],pred,score)})
        print(name,'완료')
    return pd.DataFrame(rows).sort_values(['pr_auc','f1'],ascending=False).reset_index(drop=True)
def compare_multiclass(train,dev,label):
    rows=[]
    for name,model in binary_candidates().items():
        model.fit(train.clean_text,train[label]); pred=model.predict(dev.clean_text)
        p,r,f1,_=precision_recall_fscore_support(dev[label],pred,average='macro',zero_division=0)
        rows.append({'모델':name,'accuracy':accuracy_score(dev[label],pred),'macro_precision':p,'macro_recall':r,'macro_f1':f1})
    return pd.DataFrame(rows).sort_values('macro_f1',ascending=False).reset_index(drop=True)

## 9. 정상상담 vs 보이스피싱

In [ ]:
# 12. 검증 세트에서 알고리즘과 길이·비율 민감도 비교
base_compare=compare_binary(train_1to3,dev_original)
display(base_compare)
best_detection_name=base_compare.iloc[0]['모델']
sensitivity_rows=[]
for exp_name,train_data,dev_data in [
 ('원본_1대3',train_1to3,dev_original),('원본_1대1',train_1to1,dev_original),('길이보정_1대1',train_length_1to1,dev_length)]:
    model=clone(binary_candidates()[best_detection_name]); model.fit(train_data.clean_text,train_data.fraud_label)
    pred=model.predict(dev_data.clean_text); score=positive_score(model,dev_data.clean_text,FRAUD)
    sensitivity_rows.append({'실험':exp_name,'모델':best_detection_name,**binary_scores(dev_data.fraud_label,pred,score)})
sensitivity_df=pd.DataFrame(sensitivity_rows).sort_values('pr_auc',ascending=False)
display(sensitivity_df)
base_compare.to_csv(REPORT_ROOT/'정상사기_모델비교_검증.csv',index=False,encoding='utf-8-sig')
sensitivity_df.to_csv(REPORT_ROOT/'정상사기_비율_길이_민감도.csv',index=False,encoding='utf-8-sig')
print('주의: 민감도 실험은 편향 확인용이며 최종 테스트 결과로 모델을 다시 고르지 않습니다.')

In [ ]:
# 13. 선택된 모델을 TRAIN+DEV로 재학습한 뒤 최종 TEST를 한 번만 평가
final_train=sample_ratio(det[det.ml_split.isin(['TRAIN','DEV'])],3)
final_test=det[det.ml_split.eq('TEST')].copy()
best_detection=clone(binary_candidates()[best_detection_name])
best_detection.fit(final_train.clean_text,final_train.fraud_label)
det_pred=best_detection.predict(final_test.clean_text); det_score=positive_score(best_detection,final_test.clean_text,FRAUD)
detection_test=binary_scores(final_test.fraud_label,det_pred,det_score)
display(pd.DataFrame([detection_test]))
print(classification_report(final_test.fraud_label,det_pred,zero_division=0))
det_result=final_test[['conversation_id','group_id','fraud_label','clean_text']].copy()
det_result['예측']=det_pred; det_result['보이스피싱점수']=det_score; det_result['정답여부']=det_result.fraud_label.eq(det_pred)
det_result.to_csv(PRED_ROOT/'정상사기_최종테스트_예측.csv',index=False,encoding='utf-8-sig')
joblib.dump(best_detection,MODEL_ROOT/'fraud_detection_best_model.joblib')
labels=[NORMAL,FRAUD]; cm=confusion_matrix(final_test.fraud_label,det_pred,labels=labels)
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['정상','보이스피싱'],yticklabels=['정상','보이스피싱'])
plt.xlabel('예측'); plt.ylabel('실제'); plt.title('정상 vs 보이스피싱 최종 테스트')
plt.tight_layout(); plt.savefig(REPORT_ROOT/'정상사기_혼동행렬.png',dpi=160); plt.show()

## 10. 보이스피싱 유형 분류

In [ ]:
# 14. 대출사기형 vs 수사기관사칭형
fraud_type=type_df[type_df.supervised_target.isin(['LOAN_FRAUD','INSTITUTION_IMPERSONATION'])].copy()
# fraud_type_ml에는 group_id가 없으므로 원본 통화 ID인 file_id를 분리 그룹으로 사용합니다.
assert 'file_id' in fraud_type.columns, '원본 통화 분리에 필요한 file_id가 없습니다.'
assert fraud_type['file_id'].notna().all(), 'file_id 결측값이 있습니다.'
fraud_type['group_id']=fraud_type['file_id'].astype(str)
fraud_type['ml_split']=fraud_type.group_id.map(stratified_group_parts(fraud_type,'supervised_target'))
display(check_split(fraud_type,'supervised_target'))
tr=fraud_type[fraud_type.ml_split.eq('TRAIN')]; dv=fraud_type[fraud_type.ml_split.eq('DEV')]; te=fraud_type[fraud_type.ml_split.eq('TEST')]
type_compare=compare_multiclass(tr,dv,'supervised_target'); display(type_compare)
best_type_name=type_compare.iloc[0]['모델']; best_type=clone(binary_candidates()[best_type_name])
best_type.fit(fraud_type[fraud_type.ml_split.isin(['TRAIN','DEV'])].clean_text,fraud_type[fraud_type.ml_split.isin(['TRAIN','DEV'])].supervised_target)
type_pred=best_type.predict(te.clean_text)
p,r,type_f1,_=precision_recall_fscore_support(te.supervised_target,type_pred,average='macro',zero_division=0)
type_test={'accuracy':accuracy_score(te.supervised_target,type_pred),'macro_precision':p,'macro_recall':r,'macro_f1':type_f1}
display(pd.DataFrame([type_test])); print(classification_report(te.supervised_target,type_pred,zero_division=0))
type_result=te[['case_id','group_id','supervised_target','clean_text']].copy(); type_result['예측']=type_pred; type_result['정답여부']=type_result.supervised_target.eq(type_pred)
type_result.to_csv(PRED_ROOT/'사기유형_최종테스트_예측.csv',index=False,encoding='utf-8-sig')
type_compare.to_csv(REPORT_ROOT/'사기유형_모델비교.csv',index=False,encoding='utf-8-sig')
joblib.dump(best_type,MODEL_ROOT/'fraud_type_best_model.joblib')

## 11. 전체·부분 구간 탐지

In [ ]:
# 15. 전체 대화와 임의 구간 탐지
split_lookup=det[['group_id','ml_split']].drop_duplicates(); segment=segment_df.copy(); segment['group_id']=segment.group_id.astype(str)
segment=segment.merge(split_lookup,on='group_id',how='inner'); display(check_split(segment,'fraud_label'))
tr=segment[segment.ml_split.eq('TRAIN')]; dv=segment[segment.ml_split.eq('DEV')]; te_seg=segment[segment.ml_split.eq('TEST')]
segment_compare=compare_binary(sample_ratio(tr,3),dv); display(segment_compare)
best_segment_name=segment_compare.iloc[0]['모델']; best_segment=clone(binary_candidates()[best_segment_name])
segment_train_dev=sample_ratio(segment[segment.ml_split.isin(['TRAIN','DEV'])],3)
best_segment.fit(segment_train_dev.clean_text,segment_train_dev.fraud_label)
seg_pred=best_segment.predict(te_seg.clean_text); seg_score=positive_score(best_segment,te_seg.clean_text,FRAUD)
segment_result=te_seg.copy(); segment_result['예측']=seg_pred; segment_result['보이스피싱점수']=seg_score; segment_result['정답여부']=segment_result.fraud_label.eq(seg_pred)
rows=[]
for (scope,pos),g in segment_result.groupby(['sample_scope','window_position']):
    if g.fraud_label.nunique()<2: continue
    rows.append({'표본범위':scope,'구간위치':pos,'건수':len(g),**binary_scores(g.fraud_label,g['예측'],g['보이스피싱점수'])})
position_df=pd.DataFrame(rows); display(position_df)
segment_result.to_csv(PRED_ROOT/'전체부분구간_최종테스트_예측.csv',index=False,encoding='utf-8-sig')
position_df.to_csv(REPORT_ROOT/'전체부분구간_위치별성능.csv',index=False,encoding='utf-8-sig')
joblib.dump(best_segment,MODEL_ROOT/'segment_detection_best_model.joblib')

## 12. 유사 사건 군집분석

In [ ]:
# 16. K-means·MiniBatch K-means·계층 군집 비교
vectorizer=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.95,max_features=MAX_FEATURES,sublinear_tf=True)
matrix=vectorizer.fit_transform(cluster_df.clean_text)
n_components=max(2,min(50,matrix.shape[0]-1,matrix.shape[1]-1))
svd=TruncatedSVD(n_components=n_components,random_state=SEED); features=svd.fit_transform(matrix)
rows=[]; outputs={}
for k in range(2,min(8,len(cluster_df)-1)+1):
    candidates={'kmeans':KMeans(n_clusters=k,n_init=20,random_state=SEED),
                'minibatch_kmeans':MiniBatchKMeans(n_clusters=k,n_init=20,batch_size=256,random_state=SEED),
                'agglomerative':AgglomerativeClustering(n_clusters=k)}
    for name,model in candidates.items():
        labels=model.fit_predict(features); sil=silhouette_score(features,labels); stability=np.nan
        if name!='agglomerative':
            other=clone(model).set_params(random_state=SEED+1); stability=adjusted_rand_score(labels,other.fit_predict(features))
        rows.append({'알고리즘':name,'군집수':k,'silhouette':sil,'seed_stability_ari':stability}); outputs[(name,k)]=(model,labels)
cluster_compare=pd.DataFrame(rows).sort_values(['silhouette','seed_stability_ari'],ascending=False); display(cluster_compare.head(12))
best_cluster_row=cluster_compare.iloc[0]; best_cluster_key=(best_cluster_row['알고리즘'],int(best_cluster_row['군집수']))
best_cluster_model,cluster_labels=outputs[best_cluster_key]
cluster_result=cluster_df.copy(); cluster_result['군집ID']=cluster_labels
cluster_result.to_csv(PRED_ROOT/'유사사건_군집결과.csv',index=False,encoding='utf-8-sig')
cluster_compare.to_csv(REPORT_ROOT/'군집모델_비교.csv',index=False,encoding='utf-8-sig')
joblib.dump({'vectorizer':vectorizer,'svd':svd,'model':best_cluster_model,'algorithm':best_cluster_key[0],'cluster_count':best_cluster_key[1]},MODEL_ROOT/'case_clustering_best_model.joblib')
print('군집은 정답 분류가 아니므로 정확도라고 부르지 않습니다.')

## 13. 최종 보고서와 자동 검증

In [ ]:
# 17. 결과 요약·한계·실행기록 저장
best_models=pd.DataFrame([
 ['정상상담 vs 보이스피싱',best_detection_name,'PR-AUC',detection_test['pr_auc']],
 ['보이스피싱 유형',best_type_name,'Macro F1',type_test['macro_f1']],
 ['전체·부분 구간',best_segment_name,'평균 PR-AUC',position_df.pr_auc.mean() if len(position_df) else np.nan],
 ['유사 사건 군집',f'{best_cluster_key[0]} (k={best_cluster_key[1]})','Silhouette',best_cluster_row.silhouette]
],columns=['시나리오','선정모델','최종지표','최종점수'])
display(best_models); best_models.to_csv(REPORT_ROOT/'시나리오별_최종모델.csv',index=False,encoding='utf-8-sig')
report=[
 '# 3단계 v4 머신러닝 분석 결과','', '## Summary','',
 f'- 정상·사기 모델: {best_detection_name}',f"- 최종 PR-AUC: {detection_test['pr_auc']:.4f}",f"- 보이스피싱 Recall: {detection_test['recall']:.4f}",
 f'- 사기유형 모델: {best_type_name} / Macro F1 {type_test["macro_f1"]:.4f}',
 f'- 구간탐지 모델: {best_segment_name}',f'- 군집: {best_cluster_key[0]} / k={best_cluster_key[1]}','',
 '## v4 편향 점검','',f'- 길이 보정 기준: {target_chars}자','- 1:3, 1:1, 길이보정 1:1 결과는 검증 세트에서 비교함','- 최종 테스트는 선택된 모델에 한 번만 사용함','',
 '## 해석 시 주의','',
 '- 정상상담과 보이스피싱은 출처와 원본 편집 방식이 달라 현재 점수는 두 공개 코퍼스 구분 성능입니다.',
 '- 길이 보정 후 성능이 크게 낮아지면 모델이 사기 표현 외에 텍스트 길이를 이용했을 가능성이 있습니다.',
 '- 금액 방향과 용도는 자동 추출 SILVER 라벨이며 실제 피해액을 뜻하지 않습니다.',
 '- 실제 피해 여부 정답이 없으므로 피해 발생 확률로 해석할 수 없습니다.'
]
(REPORT_ROOT/'03_ml_analysis_report_v4.md').write_text('\n'.join(report),encoding='utf-8')
manifest={'version':'v4','seed':SEED,'dataset_root':str(DATASET_ROOT),'output_root':str(OUTPUT_ROOT),
          'target_chars':target_chars,'best_models':best_models.to_dict(orient='records'),
          'group_leakage_checks_passed':True,'test_used_once_for_final_model':True,
          'saved_models':[p.name for p in MODEL_ROOT.glob('*.joblib')]}
(REPORT_ROOT/'ml_run_manifest_v4.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
assert len(list(MODEL_ROOT.glob('*.joblib')))==4
assert det.groupby('group_id').ml_split.nunique().max()==1
assert fraud_type.groupby('group_id').ml_split.nunique().max()==1
assert segment.groupby('group_id').ml_split.nunique().max()==1
print('3단계 v4 정상 완료:',OUTPUT_ROOT)